# Notebook 54 — Global `b` as a Projection of the Local Exponent Field

This notebook extends Notebook 53.

Notebook 53 established the exact local identity

`b_local(x) = x Γ(x) / ∫₀ˣ Γ(u) du`

and showed that a fitted global stretching exponent `b` is a coarse summary of the
scale-dependent local exponent field `b_local(x)`.

This notebook makes that statement more explicit:

> Can the fitted global exponent `b` be approximated as a **weighted projection**
> of the local exponent field `b_local(x)`?

We test several candidate projection rules and compare them to the fitted global exponent.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


## Core definitions

In [ ]:
x = np.linspace(1e-4, 1.0, 800)

def build_S_from_gamma(gamma_x, x):
    integral = np.cumsum(gamma_x) * (x[1] - x[0])
    return np.exp(-integral), integral

def stretched(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    popt, _ = curve_fit(
        stretched,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 3.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return a, b, pred, mse

def b_local_from_gamma(x, gamma_x, integral_x):
    eps = 1e-14
    return x * gamma_x / np.clip(integral_x, eps, None)

def normalize_weight(w, x):
    w = np.clip(np.asarray(w, dtype=float), 0.0, None)
    area = np.trapz(w, x)
    if area <= 0:
        return np.ones_like(x) / np.trapz(np.ones_like(x), x)
    return w / area

def projected_b(x, b_local, w):
    wn = normalize_weight(w, x)
    return float(np.trapz(wn * b_local, x))


## Structured rate families

In [ ]:
def gamma_corrected(x, c=1.8, m=0.9, eps=0.2, kind="linear"):
    base = c * np.power(x, m - 1.0)
    if kind == "linear":
        h = x
    elif kind == "quadratic":
        h = (x - 0.5)**2
    elif kind == "wave":
        h = np.sin(2*np.pi*x)
    elif kind == "bump":
        h = np.exp(-((x - 0.3)/0.15)**2) - 0.4*np.exp(-((x - 0.8)/0.12)**2)
    else:
        h = np.zeros_like(x)
    return base * (1.0 + eps * h)

specs = [
    ("linear", 0.20),
    ("quadratic", 0.30),
    ("wave", 0.15),
    ("bump", 0.18),
]

rows = []
for kind, eps in specs:
    gamma_x = gamma_corrected(x, c=1.8, m=0.9, eps=eps, kind=kind)
    S, I = build_S_from_gamma(gamma_x, x)
    b_local = b_local_from_gamma(x, gamma_x, I)
    a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
    rows.append({
        "kind": kind,
        "eps": eps,
        "gamma": gamma_x,
        "S": S,
        "I": I,
        "b_local": b_local,
        "b_fit": b_fit,
        "S_fit": S_fit,
        "fit_mse": mse,
    })

for row in rows:
    print(f'{row["kind"]}: b_fit={row["b_fit"]:.5f}, mse={row["fit_mse"]:.3e}')


## Plot 1 — Local exponent fields

In [ ]:
plt.figure(figsize=(8.4, 5.1))
for row in rows:
    plt.plot(x, row["b_local"], label=f'{row["kind"]}')
plt.xlabel("x")
plt.ylabel("b_local(x)")
plt.title("Local exponent fields generated by structured Γ(x)")
plt.legend()
plt.tight_layout()
plt.show()


## Candidate projection weights

In [ ]:
def make_weights(x, S, gamma_x):
    dSdx = -gamma_x * S
    log_sensitivity = np.abs(np.gradient(np.log(np.clip(-np.log(np.clip(S, 1e-14, 1.0)), 1e-14, None)),
                                         np.log(np.clip(x, 1e-14, None))))
    return {
        "uniform": np.ones_like(x),
        "x_weighted": x,
        "signal_weighted": S,
        "decay_activity": np.abs(dSdx),
        "local_sensitivity": log_sensitivity,
    }


## Compute projected exponents

In [ ]:
for row in rows:
    weights = make_weights(x, row["S"], row["gamma"])
    row["proj"] = {name: projected_b(x, row["b_local"], w) for name, w in weights.items()}
    row["weights"] = {name: normalize_weight(w, x) for name, w in weights.items()}

for row in rows:
    print("\n", row["kind"])
    print("b_fit =", row["b_fit"])
    for name, val in row["proj"].items():
        print("  ", name, "->", round(val, 6))


## Plot 2 — Fitted global b vs candidate projections

In [ ]:
proj_names = list(rows[0]["proj"].keys())
fig, axes = plt.subplots(1, len(proj_names), figsize=(18, 4.5), sharey=True)

for ax, pname in zip(axes, proj_names):
    true_vals = np.array([row["b_fit"] for row in rows], dtype=float)
    proj_vals = np.array([row["proj"][pname] for row in rows], dtype=float)
    ax.scatter(true_vals, proj_vals, s=85)
    lims = [min(np.min(true_vals), np.min(proj_vals)) - 0.01, max(np.max(true_vals), np.max(proj_vals)) + 0.01]
    ax.plot(lims, lims, linestyle="--")
    for i, row in enumerate(rows):
        ax.annotate(row["kind"], (true_vals[i], proj_vals[i]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("fitted global b")
    ax.set_title(pname)

axes[0].set_ylabel("projected b")
plt.suptitle("Global fitted b vs candidate projections of b_local(x)", y=1.03)
plt.tight_layout()
plt.show()


## Projection errors

In [ ]:
scores = {}
for pname in proj_names:
    true_vals = np.array([row["b_fit"] for row in rows], dtype=float)
    proj_vals = np.array([row["proj"][pname] for row in rows], dtype=float)
    mse = float(np.mean((true_vals - proj_vals)**2))
    mae = float(np.mean(np.abs(true_vals - proj_vals)))
    scores[pname] = {"mse": mse, "mae": mae}

scores


## Plot 3 — Weight functions for one representative case

In [ ]:
rep = rows[2]
plt.figure(figsize=(8.5, 5.2))
for name, w in rep["weights"].items():
    plt.plot(x, w, label=name)
plt.xlabel("x")
plt.ylabel("normalized weight")
plt.title(f'Candidate projection weights ({rep["kind"]} case)')
plt.legend()
plt.tight_layout()
plt.show()


## Plot 4 — How weights compress the local exponent field

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

best_names = sorted(scores.keys(), key=lambda k: scores[k]["mse"])[:4]

for ax, pname in zip(axes.ravel(), best_names):
    for row in rows:
        ax.plot(x, row["b_local"], alpha=0.25)
    ax.plot(x, rep["b_local"], color="black", linewidth=2)
    ax2 = ax.twinx()
    ax2.plot(x, rep["weights"][pname], color="red", linestyle="--")
    ax.set_title(pname)
    ax.set_xlabel("x")
    ax.set_ylabel("b_local")
    ax2.set_ylabel("weight", color="red")
    ax2.tick_params(axis='y', colors='red')

plt.tight_layout()
plt.show()


## Fitted projection family

In [ ]:
def beta_weight_family(x, p, q):
    return np.power(np.clip(x, 1e-12, 1.0), p) * np.power(np.clip(1 - x, 1e-12, 1.0), q)

p_grid = np.linspace(0.0, 4.0, 41)
q_grid = np.linspace(0.0, 4.0, 41)

best = None
best_mse = np.inf

true_vals = np.array([row["b_fit"] for row in rows], dtype=float)

for p in p_grid:
    for q in q_grid:
        pred_vals = []
        for row in rows:
            w = beta_weight_family(x, p, q)
            pred_vals.append(projected_b(x, row["b_local"], w))
        pred_vals = np.array(pred_vals, dtype=float)
        mse = float(np.mean((true_vals - pred_vals)**2))
        if mse < best_mse:
            best_mse = mse
            best = (p, q, pred_vals)

best_p, best_q, best_pred = best
print("Best (p, q) =", best_p, best_q)
print("Best MSE =", best_mse)


## Plot 5 — Best fitted projection

In [ ]:
plt.figure(figsize=(7.2, 5.0))
plt.scatter(true_vals, best_pred, s=90)
lims = [min(np.min(true_vals), np.min(best_pred)) - 0.01, max(np.max(true_vals), np.max(best_pred)) + 0.01]
plt.plot(lims, lims, linestyle="--")
for i, row in enumerate(rows):
    plt.annotate(row["kind"], (true_vals[i], best_pred[i]), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xlabel("fitted global b")
plt.ylabel("best projected b")
plt.title(f'Best fitted projection: w(x) = x^{best_p:.1f}(1-x)^{best_q:.1f}')
plt.tight_layout()
plt.show()


## Plot 6 — Best fitted weight function

In [ ]:
w_best = normalize_weight(beta_weight_family(x, best_p, best_q), x)

plt.figure(figsize=(8.0, 4.8))
plt.plot(x, w_best, linewidth=2)
plt.xlabel("x")
plt.ylabel("normalized weight")
plt.title("Best fitted projection weight")
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Global b as a projection of the local exponent field")
lines.append("")
for pname in proj_names:
    lines.append(f'- {pname}: MSE={scores[pname]["mse"]:.6e}, MAE={scores[pname]["mae"]:.6e}')
lines.append("")
lines.append(f'Best fitted projection family: w(x) = x^{best_p:.2f}(1-x)^{best_q:.2f}')
lines.append(f'Best fitted projection MSE = {best_mse:.6e}')
lines.append("")
lines.append("Interpretation:")
lines.append("- The fitted global exponent is approximated by a weighted average of b_local(x).")
lines.append("- Different weighting rules correspond to different assumptions about which scales matter.")
lines.append("- A simple parametric weight family already reproduces the fitted exponent well.")
print("\n".join(lines))


## Final conclusion

This notebook upgrades the local-exponent picture into an explicit projection formula.

The key result is:

`b ≈ ∫ w(x) b_local(x) dx`

for an appropriate weight function `w(x)`.

So the full hierarchy is now:

`Γ(x)`  
→ `b_local(x)`  
→ weighted projection  
→ global fitted exponent `b`
